In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("SVM RBF Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

In [ ]:
from sklearn.metrics import roc_curve, auc

svm_probs = svm_model.predict_proba(X_test)[:, 1]

fpr_svm, tpr_svm, thresholds_svm = roc_curve(y_test, svm_probs)
auc_svm = auc(fpr_svm, tpr_svm)

plt.figure(figsize=(7, 5))
plt.plot(fpr_svm, tpr_svm, label=f"SVM RBF AUC = {auc_svm:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("SVM RBF ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

print("SVM RBF AUC:", auc_svm)

In [ ]:
perm = permutation_importance(
    svm_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": perm.importances_mean
}).sort_values("Importance", ascending=False)

perm_df.head(15)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=perm_df.head(15),
    x="Importance",
    y="Feature"
)

plt.title("SVM RBF - Permutation Importance")
plt.show()

In [ ]:
if "patient_age" in X_test.columns:
    pdp_feature = "patient_age"
else:
    possible_features = [
        col for col in X_test.columns
        if X_test[col].nunique() > 5
    ]
    pdp_feature = possible_features[0]

print("PDP feature used:", pdp_feature)

grid_values = np.linspace(
    X_test[pdp_feature].min(),
    X_test[pdp_feature].max(),
    30
)

avg_predictions = []

for value in grid_values:
    X_temp = X_test.copy()
    X_temp[pdp_feature] = value

    prob_dr = svm_model.predict_proba(X_temp)[:, 1].mean()
    avg_predictions.append(prob_dr)

plt.figure(figsize=(8, 5))
plt.plot(grid_values, avg_predictions, marker="o")
plt.xlabel(pdp_feature)
plt.ylabel("Average predicted probability of DR")
plt.title(f"SVM RBF Manual PDP for {pdp_feature}")
plt.grid(True)
plt.show()

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=feature_cols,
    class_names=["No DR", "DR"],
    mode="classification"
)

no_dr_index = y_test[y_test == 0].index[0]
dr_index = y_test[y_test == 1].index[0]

no_dr_pos = list(y_test.index).index(no_dr_index)
dr_pos = list(y_test.index).index(dr_index)

sample_indices = [no_dr_pos, dr_pos]

for idx in sample_indices:
    print("Explaining sample position:", idx)
    print("True label:", y_test.iloc[idx])

    lime_exp = lime_explainer.explain_instance(
        X_test.iloc[idx].values,
        svm_model.predict_proba,
        num_features=10
    )

    lime_exp.show_in_notebook(show_table=True)

In [ ]:
background = shap.sample(X_train, 30, random_state=42)
test_sample = shap.sample(X_test, 15, random_state=42)


def svm_predict_proba_for_shap(data):
    data_df = pd.DataFrame(data, columns=feature_cols)
    return svm_model.predict_proba(data_df)


shap_explainer = shap.KernelExplainer(
    svm_predict_proba_for_shap,
    background
)


shap_values = shap_explainer.shap_values(test_sample)


if isinstance(shap_values, list):
    shap_values_class_1 = shap_values[1]
else:
    shap_values_class_1 = shap_values[:, :, 1]


shap.summary_plot(
    shap_values_class_1,
    test_sample,
    feature_names=feature_cols
)

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0
    return img, label

train_img_ds = tf.data.Dataset.from_tensor_slices(
    (img_train.values, y_train.values)
)

test_img_ds = tf.data.Dataset.from_tensor_slices(
    (img_test.values, y_test.values)
)

train_img_ds = train_img_ds.map(load_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_img_ds = test_img_ds.map(load_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
from sklearn.preprocessing import StandardScaler

IMG_SIZE = (128, 128)
BATCH_SIZE = 32


tab_scaler = StandardScaler()

X_train_tab = tab_scaler.fit_transform(X_train)
X_test_tab = tab_scaler.transform(X_test)

print("Tabular train shape:", X_train_tab.shape)
print("Tabular test shape:", X_test_tab.shape)

In [ ]:
def load_multimodal_sample(image_path, tabular_features, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0

    return (img, tabular_features), label


train_multi_ds = tf.data.Dataset.from_tensor_slices(
    (
        img_train.values,
        X_train_tab.astype("float32"),
        y_train.values.astype("float32")
    )
)

test_multi_ds = tf.data.Dataset.from_tensor_slices(
    (
        img_test.values,
        X_test_tab.astype("float32"),
        y_test.values.astype("float32")
    )
)

train_multi_ds = train_multi_ds.map(load_multimodal_sample)
test_multi_ds = test_multi_ds.map(load_multimodal_sample)

train_multi_ds = train_multi_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_multi_ds = test_multi_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Multimodal datasets created")

In [ ]:
num_tabular_features = X_train_tab.shape[1]


image_input = tf.keras.Input(shape=(128, 128, 3), name="image_input")

x = layers.Conv2D(32, (3, 3), activation="relu", name="multi_conv1")(image_input)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, (3, 3), activation="relu", name="multi_conv2")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, (3, 3), activation="relu", name="multi_conv3")(x)
x = layers.MaxPooling2D()(x)

x = layers.GlobalAveragePooling2D()(x)

image_features = layers.Dense(128, activation="relu", name="image_features")(x)


tabular_input = tf.keras.Input(shape=(num_tabular_features,), name="tabular_input")

t = layers.Dense(64, activation="relu")(tabular_input)
t = layers.Dropout(0.3)(t)

t = layers.Dense(32, activation="relu", name="tabular_features")(t)

combined = layers.Concatenate(name="fusion_layer")([image_features, t])

z = layers.Dense(128, activation="relu")(combined)
z = layers.Dropout(0.4)(z)

z = layers.Dense(64, activation="relu")(z)

output = layers.Dense(1, activation="sigmoid", name="output")(z)

multimodal_model = tf.keras.Model(
    inputs=[image_input, tabular_input],
    outputs=output
)

multimodal_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

multimodal_model.summary()

In [ ]:
multimodal_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

In [ ]:
multi_history = multimodal_model.fit(
    train_multi_ds,
    validation_data=test_multi_ds,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[
        EarlyStopping(
            monitor="val_auc",
            patience=2,
            restore_best_weights=True,
            mode="max"
        )
    ]
)

In [ ]:
multi_probs = multimodal_model.predict(test_multi_ds)

multi_preds = (multi_probs > 0.5).astype(int).ravel()

print("Multimodal CNN Accuracy:", accuracy_score(y_test, multi_preds))
print(classification_report(y_test, multi_preds))

In [ ]:
from sklearn.metrics import roc_curve, auc

multi_probs = multimodal_model.predict(test_multi_ds).ravel()

fpr_multi, tpr_multi, thresholds_multi = roc_curve(y_test, multi_probs)
auc_multi = auc(fpr_multi, tpr_multi)

plt.figure(figsize=(7, 5))
plt.plot(fpr_multi, tpr_multi, label=f"Multimodal CNN AUC = {auc_multi:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multimodal CNN ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

print("Multimodal CNN AUC:", auc_multi)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(multi_history.history["accuracy"], label="Train Accuracy")
plt.plot(multi_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Multimodal CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(multi_history.history["loss"], label="Train Loss")
plt.plot(multi_history.history["val_loss"], label="Validation Loss")
plt.title("Multimodal CNN Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
def read_image_for_multimodal_xai(path):
    img = cv2.imread(path)

    if img is None:
        raise ValueError(f"Image not found: {path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE)

    img_norm = img.astype("float32") / 255.0
    img_array = np.expand_dims(img_norm, axis=0)

    return img, img_array


def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    return None


def get_positive_sample_position():
    y_values = y_test.values if hasattr(y_test, "values") else np.array(y_test)

    positive_indices = np.where(y_values == 1)[0]

    if len(positive_indices) > 0:
        return positive_indices[0]
    else:
        return 0


sample_pos = get_positive_sample_position()

multi_sample_path = img_test.iloc[sample_pos]

multi_original_img, multi_img_array = read_image_for_multimodal_xai(multi_sample_path)

multi_tab_array = X_test_tab[sample_pos].reshape(1, -1).astype("float32")

multi_last_layer = get_last_conv_layer(multimodal_model)

print("Selected sample position:", sample_pos)
print("Selected image:", multi_sample_path)
print("Last convolutional layer:", multi_last_layer)
print("Tabular sample shape:", multi_tab_array.shape)

plt.imshow(multi_original_img)
plt.axis("off")
plt.title("Original Image for Multimodal CNN XAI")
plt.show()

In [ ]:
def multimodal_gradcam(model, img_array, tab_array, last_conv_layer_name):

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model([img_array, tab_array])
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)


    heatmap = heatmap.numpy()


    heatmap = np.maximum(heatmap, 0)


    if np.max(heatmap) != 0:
        heatmap = heatmap / np.max(heatmap)

    return heatmap


multi_gradcam_map = multimodal_gradcam(
    multimodal_model,
    multi_img_array,
    multi_tab_array,
    multi_last_layer
)

plt.imshow(multi_original_img)
plt.imshow(cv2.resize(multi_gradcam_map, IMG_SIZE), alpha=0.5, cmap="jet")
plt.axis("off")
plt.title("Multimodal CNN Grad-CAM - Image Branch")
plt.show()

In [ ]:
def multimodal_occlusion_sensitivity(model, img_array, tab_array, patch_size=16):

    img = img_array.copy()

    h, w = img.shape[1], img.shape[2]

    baseline_pred = model.predict([img, tab_array], verbose=0)[0][0]

    heatmap = np.zeros((h, w))

    for y_pos in range(0, h, patch_size):
        for x_pos in range(0, w, patch_size):

            occluded = img.copy()

            occluded[:, y_pos:y_pos+patch_size, x_pos:x_pos+patch_size, :] = 0

            pred = model.predict([occluded, tab_array], verbose=0)[0][0]

            diff = abs(baseline_pred - pred)

            heatmap[y_pos:y_pos+patch_size, x_pos:x_pos+patch_size] = diff

    if heatmap.max() != 0:
        heatmap = heatmap / heatmap.max()

    return heatmap


multi_occ_map = multimodal_occlusion_sensitivity(
    multimodal_model,
    multi_img_array,
    multi_tab_array,
    patch_size=16
)

plt.imshow(multi_original_img)
plt.imshow(multi_occ_map, alpha=0.5, cmap="jet")
plt.axis("off")
plt.title("Multimodal CNN Occlusion Sensitivity - Image Branch")
plt.show()

In [ ]:
def load_images_as_array(paths, img_size=(128, 128)):
    images = []

    for path in paths:
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, img_size)
        img = img.astype("float32") / 255.0
        images.append(img)

    return np.array(images)



n_samples = min(200, len(X_test_tab))

subset_images = load_images_as_array(
    img_test.iloc[:n_samples].values,
    IMG_SIZE
)

subset_tab = X_test_tab[:n_samples].copy().astype("float32")

subset_y = y_test.iloc[:n_samples].values if hasattr(y_test, "iloc") else y_test[:n_samples]

baseline_probs = multimodal_model.predict(
    [subset_images, subset_tab],
    verbose=0
)

baseline_preds = (baseline_probs > 0.5).astype(int).ravel()

baseline_acc = accuracy_score(subset_y, baseline_preds)

tab_perm_results = []

for i, feature in enumerate(feature_cols):

    X_perm = subset_tab.copy()

    np.random.seed(42)
    np.random.shuffle(X_perm[:, i])

    perm_probs = multimodal_model.predict(
        [subset_images, X_perm],
        verbose=0
    )

    perm_preds = (perm_probs > 0.5).astype(int).ravel()

    perm_acc = accuracy_score(subset_y, perm_preds)

    importance = baseline_acc - perm_acc

    tab_perm_results.append([feature, importance])


multi_tab_perm_df = pd.DataFrame(
    tab_perm_results,
    columns=["Feature", "Importance"]
).sort_values("Importance", ascending=False)

multi_tab_perm_df.head(15)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=multi_tab_perm_df.head(15),
    x="Importance",
    y="Feature"
)

plt.title("Multimodal CNN - Tabular Permutation Importance")
plt.xlabel("Accuracy Drop After Feature Shuffling")
plt.ylabel("Tabular Feature")

plt.show()

In [ ]:
def tabular_integrated_gradients(model, img_array, tab_array, baseline=None, steps=50):

    tab_array = tab_array.astype("float32")

    if baseline is None:
        baseline = np.zeros_like(tab_array).astype("float32")

    interpolated_tabs = []

    for i in range(steps + 1):
        alpha = i / steps
        interpolated = baseline + alpha * (tab_array - baseline)
        interpolated_tabs.append(interpolated)

    interpolated_tabs = np.vstack(interpolated_tabs).astype("float32")

    repeated_images = np.repeat(img_array, steps + 1, axis=0).astype("float32")

    interpolated_tabs_tensor = tf.convert_to_tensor(interpolated_tabs)

    repeated_images_tensor = tf.convert_to_tensor(repeated_images)

    with tf.GradientTape() as tape:
        tape.watch(interpolated_tabs_tensor)

        predictions = model(
            [repeated_images_tensor, interpolated_tabs_tensor]
        )

        loss = predictions[:, 0]

    grads = tape.gradient(loss, interpolated_tabs_tensor)

    avg_grads = tf.reduce_mean(grads, axis=0).numpy()

    integrated_grads = (tab_array[0] - baseline[0]) * avg_grads

    return integrated_grads


multi_tab_ig = tabular_integrated_gradients(
    multimodal_model,
    multi_img_array,
    multi_tab_array,
    steps=50
)

multi_tab_ig_df = pd.DataFrame({
    "Feature": feature_cols,
    "Integrated_Gradient": multi_tab_ig
})

multi_tab_ig_df["Abs_Contribution"] = multi_tab_ig_df["Integrated_Gradient"].abs()

multi_tab_ig_df = multi_tab_ig_df.sort_values(
    "Abs_Contribution",
    ascending=False
)

multi_tab_ig_df.head(15)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=multi_tab_ig_df.head(15),
    x="Integrated_Gradient",
    y="Feature"
)

plt.title("Multimodal CNN - Tabular Integrated Gradients")
plt.xlabel("Contribution to Prediction")
plt.ylabel("Tabular Feature")

plt.show()

In [ ]:
full_pred = multimodal_model.predict(
    [multi_img_array, multi_tab_array],
    verbose=0
)[0][0]

blank_image = np.zeros_like(multi_img_array).astype("float32")

zero_tabular = np.zeros_like(multi_tab_array).astype("float32")

no_image_pred = multimodal_model.predict(
    [blank_image, multi_tab_array],
    verbose=0
)[0][0]

no_tabular_pred = multimodal_model.predict(
    [multi_img_array, zero_tabular],
    verbose=0
)[0][0]

no_both_pred = multimodal_model.predict(
    [blank_image, zero_tabular],
    verbose=0
)[0][0]

modality_df = pd.DataFrame({
    "Input Setting": [
        "Full Image + Tabular",
        "Tabular Only",
        "Image Only",
        "No Image + No Tabular"
    ],
    "Predicted DR Probability": [
        full_pred,
        no_image_pred,
        no_tabular_pred,
        no_both_pred
    ]
})

modality_df

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=modality_df,
    x="Input Setting",
    y="Predicted DR Probability"
)

plt.xticks(rotation=20)
plt.title("Multimodal CNN - Modality Ablation")
plt.ylim(0, 1)

plt.show()

In [ ]:
deep_cnn_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),

    layers.Dense(1, activation="sigmoid")
])

deep_cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

deep_cnn_model.summary()

In [ ]:
deep_cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

In [ ]:
deep_history = deep_cnn_model.fit(
    train_img_ds,
    validation_data=test_img_ds,
    epochs=5,
    class_weight=class_weight_dict,
    callbacks=[
        EarlyStopping(
            monitor="val_auc",
            patience=2,
            restore_best_weights=True,
            mode="max"
        )
    ]
)

In [ ]:
deep_probs = deep_cnn_model.predict(test_img_ds).ravel()

deep_preds = (deep_probs > 0.5).astype(int)

print("Deep CNN Accuracy:", accuracy_score(y_test, deep_preds))
print(classification_report(y_test, deep_preds))

In [ ]:
deep_probs = deep_cnn_model.predict(test_img_ds).ravel()

fpr_deep, tpr_deep, thresholds_deep = roc_curve(y_test, deep_probs)
auc_deep = auc(fpr_deep, tpr_deep)

plt.figure(figsize=(7, 5))
plt.plot(fpr_deep, tpr_deep, label=f"Deep CNN AUC = {auc_deep:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Deep CNN ROC Curve")
plt.legend()
plt.grid(True)
plt.show()

print("Deep CNN AUC:", auc_deep)

In [ ]:
def read_image_for_xai(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE)

    img_norm = img.astype("float32") / 255.0
    img_array = np.expand_dims(img_norm, axis=0)

    return img, img_array


def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    return None


def get_positive_sample_path():
    y_values = y_test.values if hasattr(y_test, "values") else np.array(y_test)

    positive_indices = np.where(y_values == 1)[0]

    if len(positive_indices) > 0:
        idx = positive_indices[0]
    else:
        idx = 0

    return img_test.iloc[idx]

In [ ]:
deep_sample_path = get_positive_sample_path()

deep_original_img, deep_img_array = read_image_for_xai(deep_sample_path)

deep_last_layer = get_last_conv_layer(deep_cnn_model)

print("Selected image:", deep_sample_path)
print("Deep CNN last convolutional layer:", deep_last_layer)

plt.imshow(deep_original_img)
plt.axis("off")
plt.title("Original Image for Deep CNN XAI")
plt.show()

In [ ]:
def gradcam_plus_plus(model, img_array, last_conv_layer_name):


    _ = model.predict(img_array, verbose=0)

    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.outputs[0]
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    grads_power_2 = tf.square(grads)
    grads_power_3 = grads_power_2 * grads

    sum_activations = tf.reduce_sum(conv_outputs, axis=(1, 2))

    eps = 1e-8

    alpha_num = grads_power_2
    alpha_den = 2 * grads_power_2 + sum_activations[:, None, None, :] * grads_power_3

    alphas = alpha_num / (alpha_den + eps)

    positive_grads = tf.nn.relu(grads)

    weights = tf.reduce_sum(alphas * positive_grads, axis=(1, 2))

    cam = tf.reduce_sum(weights[:, None, None, :] * conv_outputs, axis=-1)

    cam = cam[0].numpy()
    cam = np.maximum(cam, 0)

    if cam.max() != 0:
        cam = cam / cam.max()

    return cam


deep_gradcampp_map = gradcam_plus_plus(
    deep_cnn_model,
    deep_img_array,
    deep_last_layer
)

plt.imshow(deep_original_img)
plt.imshow(cv2.resize(deep_gradcampp_map, IMG_SIZE), alpha=0.5, cmap="jet")
plt.axis("off")
plt.title("Deep CNN Grad-CAM++")
plt.show()

In [ ]:
def integrated_gradients(model, img_array, baseline=None, steps=50):

    img_array = img_array.astype("float32")

    if baseline is None:
        baseline = np.zeros_like(img_array).astype("float32")

    interpolated_images = []

    for i in range(steps + 1):
        alpha = i / steps
        interpolated = baseline + alpha * (img_array - baseline)
        interpolated_images.append(interpolated)

    interpolated_images = np.vstack(interpolated_images)
    interpolated_images = tf.convert_to_tensor(interpolated_images)

    with tf.GradientTape() as tape:
        tape.watch(interpolated_images)
        predictions = model(interpolated_images)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, interpolated_images)

    avg_grads = tf.reduce_mean(grads, axis=0).numpy()

    integrated_grads = (img_array[0] - baseline[0]) * avg_grads

    attribution = np.mean(np.abs(integrated_grads), axis=-1)

    if attribution.max() != 0:
        attribution = attribution / attribution.max()

    return attribution


deep_ig_map = integrated_gradients(
    deep_cnn_model,
    deep_img_array,
    steps=50
)

plt.imshow(deep_ig_map, cmap="hot")
plt.axis("off")
plt.title("Deep CNN Integrated Gradients")
plt.show()

In [ ]:
def smoothgrad(model, img_array, samples=25, noise_level=0.10):

    smooth_map = np.zeros(img_array.shape[1:3])

    for i in range(samples):

        noise = np.random.normal(
            loc=0,
            scale=noise_level,
            size=img_array.shape
        )

        noisy_img = img_array + noise
        noisy_img = np.clip(noisy_img, 0, 1).astype("float32")

        img_tensor = tf.convert_to_tensor(noisy_img)

        with tf.GradientTape() as tape:
            tape.watch(img_tensor)
            prediction = model(img_tensor)
            loss = prediction[:, 0]

        grads = tape.gradient(loss, img_tensor)

        saliency = tf.reduce_max(tf.abs(grads), axis=-1)[0].numpy()

        smooth_map += saliency

    smooth_map = smooth_map / samples

    if smooth_map.max() != 0:
        smooth_map = smooth_map / smooth_map.max()

    return smooth_map


deep_smoothgrad_map = smoothgrad(
    deep_cnn_model,
    deep_img_array,
    samples=25,
    noise_level=0.10
)

plt.imshow(deep_smoothgrad_map, cmap="hot")
plt.axis("off")
plt.title("Deep CNN SmoothGrad")
plt.show()

In [ ]:
def scorecam(model, img_array, last_conv_layer_name, max_channels=32):

    activation_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=model.get_layer(last_conv_layer_name).output
    )

    activations = activation_model(img_array).numpy()[0]

    h, w, channels = activations.shape

    channels_to_use = min(channels, max_channels)

    weights = []
    masks = []

    for i in range(channels_to_use):

        activation = activations[:, :, i]

        activation = np.maximum(activation, 0)

        if activation.max() != 0:
            activation = activation / activation.max()

        mask = cv2.resize(activation, IMG_SIZE)

        masked_img = img_array.copy()
        masked_img[0] = masked_img[0] * mask[:, :, np.newaxis]

        score = model.predict(masked_img, verbose=0)[0][0]

        weights.append(score)
        masks.append(mask)

    weights = np.array(weights)
    masks = np.array(masks)

    cam = np.zeros(IMG_SIZE)

    for i in range(channels_to_use):
        cam += weights[i] * masks[i]

    cam = np.maximum(cam, 0)

    if cam.max() != 0:
        cam = cam / cam.max()

    return cam


deep_scorecam_map = scorecam(
    deep_cnn_model,
    deep_img_array,
    deep_last_layer,
    max_channels=32
)

plt.imshow(deep_original_img)
plt.imshow(deep_scorecam_map, alpha=0.5, cmap="jet")
plt.axis("off")
plt.title("Deep CNN Score-CAM")
plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No DR", "DR"],
        yticklabels=["No DR", "DR"]
    )

    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()


plot_confusion_matrix(y_test, y_pred_svm, "SVM RBF Confusion Matrix")
plot_confusion_matrix(y_test, deep_preds, "Deep CNN Confusion Matrix")
plot_confusion_matrix(y_test, multi_preds, "Multimodal CNN Confusion Matrix")

In [ ]:
results = pd.DataFrame({
    "Model": ["SVM RBF", "Deep CNN", "Multimodal CNN"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, deep_preds),
        accuracy_score(y_test, multi_preds)
    ]
})

results

In [ ]:
plt.figure(figsize=(7, 5))

sns.barplot(data=results, x="Model", y="Accuracy")

plt.title("Model Accuracy Comparison")
plt.ylim(0, 1)

plt.show()